# Optional DEC experiment on MNIST

This notebook implements a replication attempt of the MNIST experiment in
[Xie et al. (2016), Sections 4.2–4.4](https://proceedings.mlr.press/v48/xieb16.pdf).
Model structure and mathematical correctness are the project's primary acceptance
criteria; the benchmark is an additional check of the implemented procedure.

**Saved execution reviewed.** All five code cells executed in order on CPU,
with no saved error outputs. Autoencoder pretraining completed, and DEC reached
its assignment-change stopping criterion after **8,160 updates**. Final matched
clustering accuracy is **87.23%**. The commentary below describes this saved run
and should be revisited if its configuration or outputs change.

The paper reports clustering experiments rather than software unit tests. Here
we compare KMeans on pretrained embeddings with DEC using matched clustering
accuracy (paper Eq. 10), with ARI/NMI as extra diagnostics. This is one seed
(**42**) and one target-refresh interval (**160 updates**), not a reproduction
of all nine settings in Table 2. The full autoencoder pretraining budget is used.

The data cell fetches or reuses cached MNIST when executed. See the
[implementation scope](../docs/dec_paper_alignment.md) for differences from the
original Caffe solver; this notebook records the completed run after the
reconstruction-loss correction.


In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import torch
from sklearn.datasets import fetch_openml

from dec_model import DEVICE, initialize_clusters
from dec_pretraining import pretrain_autoencoder
from dec_training import train_dec
from latent_space_tools import compare_clustering_stages

np.random.seed(42)
torch.manual_seed(42)
print("Device:", DEVICE)
print("Seed: 42; full paper pretraining budget; execution may be lengthy.")

Device: cpu
Seed: 42; full paper pretraining budget; execution may be lengthy.


## Data and experimental population

Load OpenML's MNIST version 1: 70,000 images with 784 pixels. Retain all pixels
and multiply their 0–255 values by 0.02, following the authors'
[`make_mnist_data`](https://github.com/piiswrong/dec/blob/master/dec/dec.py).
The first 60,000 rows are the conventional MNIST training partition used for
autoencoder pretraining; clustering then uses all 70,000 rows, as in their code.
No label enters either fitting phase. This is transductive clustering of the
combined population, not a held-out classification test.

In [2]:
mnist = fetch_openml(
    "mnist_784", version=1, as_frame=False,
    data_home=str(PROJECT_ROOT / "data" / "raw" / "openml"),
)
X = mnist.data.astype(np.float32) * 0.02
# Check the dataset boundary before relying on the conventional partition.
assert X.shape == (70000, 784), X.shape
assert np.isfinite(X).all() and X.min() >= 0 and X.max() <= 5.1 + 1e-6
X_pretrain = X[:60000]
print("Autoencoder population:", X_pretrain.shape)
print("Clustering population:", X.shape)
print("Mean squared pixel value:", float(np.mean(X ** 2)))

Autoencoder population: (60000, 784)
Clustering population: (70000, 784)
Mean squared pixel value: 2.9204556941986084


## Greedy pretraining and full-autoencoder fine-tuning

Use `784–500–500–2000–10`, 50,000 updates per shallow layer, 20% dropout,
then 100,000 updates without dropout. Initialize weights with Gaussian standard
deviation 0.01. Use batch size 256, SGD with momentum 0.9 and zero weight decay;
start at learning rate 0.1 and divide by 10 every 20,000 updates, resetting the
schedule for each layer and the full autoencoder. Both encoder and decoder
weights carry over into fine-tuning. Discard the decoder before clustering.
Momentum ramps from 0 to 0.9 over the first 1,000 updates of each phase,
following the authors' pretraining solver.

**Loss normalization matters.** The authors' bundled CPU and GPU loss layers
divide squared error by twice the total element count (`N * d`). Training uses
`0.5 * mse_loss(..., reduction="mean")` in every shallow layer and full-autoencoder
phase. The final reconstruction diagnostics report ordinary MSE, without the
factor of one half. See the [authors' loss source](https://github.com/piiswrong/dec/blob/e551e01f90a3d67d2ef9c90e968c8faf5d8f857d/caffe/src/caffe/layers/euclidean_loss_layer.cpp).

The saved execution below completes all phases with this reduction.
Non-finite loss and gradient checks remain enabled.


In [3]:
encoder = pretrain_autoencoder(
    X_pretrain,
    layer_dims=[500, 500, 2000, 10],
    batch_size=256,
    pretrain_steps=50000,
    finetune_steps=100000,
    lr=0.1,
    momentum=0.9,
    momentum_burnin_steps=1000,
    lr_step_size=20000,
    lr_gamma=0.1,
    dropout_rate=0.2,
    device=DEVICE,
)


Starting Autoencoder Pretraining
Initial input shape: (60000, 784)
Input and hidden dropout probability: 0.2

Pretraining Layer 1: 784 -> 500
   Step      1 | Batch reconstruction objective: 1.506298
   Step  10000 | Batch reconstruction objective: 0.167629
   Step  20000 | Batch reconstruction objective: 0.154953
   Step  30000 | Batch reconstruction objective: 0.151115
   Step  40000 | Batch reconstruction objective: 0.154164
   Step  50000 | Batch reconstruction objective: 0.149344
Output shape after Layer 1: torch.Size([60000, 500])

Pretraining Layer 2: 500 -> 500
   Step      1 | Batch reconstruction objective: 0.653751
   Step  10000 | Batch reconstruction objective: 0.081475
   Step  20000 | Batch reconstruction objective: 0.068408
   Step  30000 | Batch reconstruction objective: 0.071288
   Step  40000 | Batch reconstruction objective: 0.064690
   Step  50000 | Batch reconstruction objective: 0.066052
Output shape after Layer 2: torch.Size([60000, 500])

Pretraining Layer 3: 

## Read the pretraining result

All four shallow layers complete their 50,000 updates, and the full autoencoder
completes 100,000 updates. The displayed objectives remain finite. Within each
shallow layer, the final displayed objective is below its initial value. Losses
across different layers reconstruct different representations and are not directly
comparable measures of representation quality.

The final full-sample reconstruction MSE is **0.242429**, compared with
**1.749207** for a constant per-pixel mean predictor on the same 60,000 images:
an approximately **86.14% reduction**. This demonstrates reconstruction beyond
the constant baseline on the pretraining sample; it is not a held-out result.

The last fine-tuning log entry, **0.122862**, is half-MSE for one mini-batch.
The final **0.242429** is ordinary MSE evaluated over the complete pretraining
sample with one fixed model. Their different reduction and evaluation population
explain why they should not be compared as if they were the same statistic.
Batch sampling also explains why the printed losses need not decrease at every
checkpoint.

The ten-sample encoder preview shows nonzero spread in every latent coordinate.
This rules out a constant embedding for those ten examples, but does not establish
full-sample effective rank or independent semantic factors. Clustering performance
is evaluated separately below.

## Initialize clusters and optimize DEC

KMeans uses 20 restarts and seed 42 through `initialize_clusters`. DEC uses
Student-t alpha 1 and constant learning rate 0.01. The refresh interval of
160 mini-batch updates follows the authors' MNIST script default. The paper
compares intervals `10 × 2**i` for `i = 0, ..., 8`; this notebook does not run
that sweep. The 20,000-update cap is a local runtime limit, not a paper parameter.
A run that reaches it without satisfying the stopping criterion should be
reported as budget-limited rather than converged.


In [4]:
Z_initial, centers, initial_labels = initialize_clusters(
    encoder, X, k=10, batch_size=256, device=DEVICE,
)
result = train_dec(
    encoder, X, centers,
    batch_size=256,
    max_steps=20000,
    update_interval=160,
    lr=0.01,
    momentum=0.9,
    alpha=1.0,
    tol=0.001,
    device=DEVICE,
)
print("Converged:", result.converged)
print("Optimizer updates:", len(result.loss_history))
print("Assignment change at last refresh:", result.label_change)


Running KMeans clustering on latent features...
Latent shape Z: (70000, 10)
Initial centroids mu: (10, 10)
Cluster size distribution: [5420 6461 6545 6853 9273 7544 6254 6816 7958 6876]
Step 1 | Batch mean KL: 0.207546
Step 160 | Batch mean KL: 0.036054
Step 160 | Changed assignments: 0.039129
Step 320 | Batch mean KL: 0.082523
Step 320 | Changed assignments: 0.023443
Step 480 | Batch mean KL: 0.100807
Step 480 | Changed assignments: 0.015100
Step 640 | Batch mean KL: 0.095214
Step 640 | Changed assignments: 0.010700
Step 800 | Batch mean KL: 0.092571
Step 800 | Changed assignments: 0.007800
Step 960 | Batch mean KL: 0.088624
Step 960 | Changed assignments: 0.006043
Step 1120 | Batch mean KL: 0.085663
Step 1120 | Changed assignments: 0.004943
Step 1280 | Batch mean KL: 0.086955
Step 1280 | Changed assignments: 0.005186
Step 1440 | Batch mean KL: 0.067029
Step 1440 | Changed assignments: 0.004786
Step 1600 | Batch mean KL: 0.071512
Step 1600 | Changed assignments: 0.003314
Step 1760 | 

## Read the DEC optimization result

The pretrained embedding has shape **(70,000, 10)**, with **(10, 10)** initial
centroids. KMeans initially occupies all ten clusters, with sizes ranging from
**5,420 to 9,273**. These counts describe initialization; final cluster sizes are
not printed in this notebook.

DEC stops after **8,160 mini-batch updates**, before the 20,000-update cap.
At the final refresh, **0.0009 = 0.09%** of hard assignments change relative to
the previous refresh: **63 of 70,000 images**, below the **0.1%** threshold.
`Converged: True` therefore correctly indicates that the specified assignment
stability criterion was met. It does not establish a global optimum or guarantee
that the stable clusters match digit identities.

The displayed batch KL decreases overall from **0.207546** to **0.028123**, with
fluctuations. Each value concerns a mini-batch, and the target distribution is
refreshed periodically, so this is not a monotonic trace of one fixed full-sample
objective. The stopping decision uses assignment changes, not a KL threshold.

## Evaluate after fitting

The paper's MNIST reference values are **81.84%** for AE + KMeans (Table 3) and
**84.30%** for DEC (Tables 2/3). Table 2 reports the best DEC result across nine
refresh settings; this notebook uses only one. The values are reference points,
not assertions or acceptance thresholds. Labels are accessed for evaluation
below and do not enter the pretraining or DEC optimization objectives.


In [5]:
y = mnist.target.astype(np.int64)
metrics, details = compare_clustering_stages(
    y,
    {"AE + KMeans": initial_labels, "DEC": result.labels},
)
metrics["paper_reference_ACC"] = [0.8184, 0.8430]
metrics

,stage,ARI,NMI,matched_accuracy,paper_reference_ACC
0,AE + KMeans,0.704602,0.753570,0.817914,0.8184
1,DEC,0.816176,0.847446,0.872329,0.8430


## Results and interpretation

| Stage | ARI | NMI | Matched accuracy | Paper reference accuracy |
| --- | ---: | ---: | ---: | ---: |
| AE + KMeans | 0.7046 | 0.7536 | 81.79% | 81.84% |
| DEC | 0.8162 | 0.8474 | 87.23% | 84.30% |

DEC refinement improves matched accuracy by **5.44 percentage points**, ARI by
**0.1116**, and NMI by **0.0939** relative to its own pretrained initialization.
The agreement of all three metrics supports improved correspondence between
clusters and digit labels in this run. Hungarian matched accuracy uses the best
one-to-one cluster-to-digit mapping; it is not the accuracy of a trained digit
classifier.

AE + KMeans is approximately **0.05 percentage points below** the paper's
reference, while DEC is **2.93 percentage points above** its reference. This
numerical comparison does not establish that the port improves on the published
method: it is a single realization, with no repeated-seed uncertainty estimate,
and the original experiment uses a different implementation and a nine-setting
comparison. The remaining solver, dropout-boundary and batch-order differences
are documented in the implementation scope. These outputs do not isolate which
differences account for the score gap.

**Conclusion for this project.** The corrected implementation now executes the
complete MNIST workflow: greedy pretraining, full-autoencoder fine-tuning,
KMeans initialization and joint DEC refinement. Reconstruction improves over the
constant baseline, DEC meets its specified stopping criterion, and the final
clustering improves over initialization. Together with the separate equation and
gradient checks, this provides evidence for the project's structural-replication
objective. It does not prove exact equivalence to the authors' Caffe execution.

The evaluation is transductive over the same 70,000 images used for clustering,
not a held-out generalization test. The frozen-encoder ablation, imbalance
experiment, other datasets and full refresh-interval sweep remain outside this
notebook. Further performance tuning is not required for the stated project goal.
